In [ ]:
using CSV, DataFrames, Plots
gr()

case_dir = joinpath(@__DIR__, "case_final")

# One entry per snapshot, taken from a LedaFlow profile export at a given instant.
profiles = [
    ("5600s",  "pressure_profile_5600.csv", :dashdotdot, :circle),
    ("20000s", "pressure_profile_20000.csv", :dot, :rect),
    ("38000s", "pressure_profile_38000.csv", :solid, :x),
]

# Column layout (1-based) in the LedaFlow export. 
# Each pipe is stored as a (position [m], pressure [bara]) column pair
series = [
    (label = "Mainline", pos = 5, val = 6, color = :steelblue,  ls = :solid, length=39000),
    (label = "Line 1",   pos = 1, val = 2, color = :darkorange, ls = :solid, length=9000),
    (label = "Well 1",     pos = 7, val = 8, color = :crimson,    ls = :solid, length=1200),
]

3-element Vector{@NamedTuple{label::String, pos::Int64, val::Int64, color::Symbol, ls::Symbol, length::Int64}}:
 (label = "Mainline", pos = 5, val = 6, color = :steelblue, ls = :solid, length = 39000)
 (label = "Line 1", pos = 1, val = 2, color = :darkorange, ls = :solid, length = 9000)
 (label = "Well 1", pos = 7, val = 8, color = :crimson, ls = :solid, length = 1200)

In [5]:
# Read a LedaFlow profile export: ';' delimited, ',' decimal. Row 1 is the
# case name and row 2 is the units/name header, so data starts at row 3. The
# pipes have different lengths, so the short columns come in padded with
# `missing`.
read_profile(path) = CSV.read(path, DataFrame;
    header = false,
    skipto = 3,
    delim = ';',
    decimal = ',',
    types = Float64,
    silencewarnings = true,
)

# Pull out a (position, pressure) column pair, dropping the trailing missings.
function getxy(df, cpos, cval, length)
    x = df[!, cpos]; y = df[!, cval]
    # LedaFlow reports position from the pipe inlet; flip it so that 0 is the pipe
    # outlet and the x-axis runs in the direction of decreasing pressure.
    x = length .- x
    keep = .!(ismissing.(x) .| ismissing.(y))
    return Float64.(x[keep]), Float64.(y[keep])
end

# One subplot: every series plotted vs distance. 
# legend lives in a separate panel, so individual subplots draw no legend.
function profile_subplot(df, title; show_xlabel = false)
    p = plot(; title = title, legend = false,
             ylabel = "Pressure (bar)",
             xlabel = show_xlabel ? "Distance (m)" : "")
    for s in series
        x, y = getxy(df, s.pos, s.val, s.length)
        plot!(p, x, y; color = s.color, ls = s.ls, lw = 3)
    end
    return p
end

profile_subplot (generic function with 1 method)

In [ ]:
default(
    legendfont=(12, "Computer Modern"), 
    titlefont=(16, "Computer Modern", :bold), 
    tickfont=(14, "Computer Modern"), 
    guidefont=(14, "Computer Modern"),
    lw=3,
    yformatter=:plain,
    xformatter=:plain,
    grid=false,
    xlabel="Distance from pipe outlet (fraction of pipe length)",
    dpi=1500
)

# Create the figure FIRST, then plot! into it. 
fig = plot(left_margin = 5Plots.mm, right_margin = 10Plots.mm,
           ylabel = "Pressure (bar)", legend=:bottomright,
           size=(800, 600), lw=1.5)

for (i, (time, f, style, marker)) in enumerate(profiles)
    df = read_profile(joinpath(case_dir, f))
    for s in series
        x, y = getxy(df, s.pos, s.val, s.length)   # flips x, drops missings
        x = x./s.length
        # Normalise by pipe length so pipes of very different lengths overlay on one axis
        plot!(fig, x, y; color = s.color, ls = style, 
              label = string(s.label)*" "*"("*time*")" )
    end
end
savefig(fig, joinpath(@__DIR__, "pressure_profile_single_plot.png"))

fig
